# Observatorio de Educacion - Analisis por Comuna, Correlacion e Indice

**Objetivo:** Cruzar fuentes por comuna, analizar correlaciones entre indicadores educativos
y evaluar la viabilidad de calcular el ICET (Indice de Calidad Educativa Territorial).

**Fuentes cruzadas:**
- Matricula 2026 (hoja Por comuna): oficial, no oficial, total
- Detalle por sede (docentes y equipos): 337 sedes con matricula, equipos, ratio
- Info geografica: asignacion de comuna a cada sede

**Graficos incluidos:** Barras por comuna, composicion sectorial, scatter, heatmap correlacion.

## Celda 0 - Configuracion del entorno

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/Observatorio_de_Educacion'
    if not os.path.exists(REPO_DIR):
        os.system('git clone https://github.com/j0rg3c45/Observatorio_de_Educacion.git ' + REPO_DIR)
    os.chdir(REPO_DIR)
    DATA_PATH = 'data/Fuentes de datos'
else:
    DATA_PATH = '../data/Fuentes de datos'
print(f'Entorno: {"Colab" if IN_COLAB else "Local"}')
print(f'Datos: {DATA_PATH}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.size'] = 11
pd.set_option('display.float_format', '{:,.2f}'.format)

def fmt_num(v):
    if abs(v) >= 1_000_000: return f'{v/1_000_000:.1f}M'
    elif abs(v) >= 1_000: return f'{v/1_000:.1f}K'
    else: return f'{v:,.0f}'

def fmt_eje(ax, eje='x'):
    def _f(x, pos):
        if abs(x)>=1e6: return f'{x/1e6:.1f}M'
        elif abs(x)>=1e3: return f'{x/1e3:.0f}K'
        else: return f'{x:.0f}'
    if eje=='x': ax.xaxis.set_major_formatter(mticker.FuncFormatter(_f))
    else: ax.yaxis.set_major_formatter(mticker.FuncFormatter(_f))

print('Librerias cargadas')

---
## 1. Matricula por comuna

In [ ]:
# Cargar matricula por comuna
ruta_mat = os.path.join(DATA_PATH, 'Reporte de matr\u00edcula', '01_Matricula_2026.xlsx')
if not os.path.exists(ruta_mat):
    ruta_mat = os.path.join(DATA_PATH, 'Reporte de matricula', '01_Matricula_2026.xlsx')

df_comuna = pd.read_excel(ruta_mat, sheet_name='Por comuna')

# Filtrar comunas urbanas (1-22)
df_cu = df_comuna[df_comuna['comuna'].str.contains('Comuna', na=False)].copy()
df_cu['num_comuna'] = df_cu['comuna'].str.extract(r'(\d+)').astype(int)
df_cu = df_cu.sort_values('num_comuna').reset_index(drop=True)
df_cu['pct_oficial'] = (df_cu['Oficial'] / df_cu['Total'] * 100).round(1)

print(f'Comunas urbanas: {len(df_cu)}')
print(f'Matricula total urbana: {df_cu["Total"].sum():,}')
df_cu[['comuna','Oficial','No oficial','Total','pct_oficial']]

In [ ]:
# Grafico: Matricula total por comuna
fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(df_cu['comuna'], df_cu['Total'],
              color=sns.color_palette('Set2', len(df_cu)))
ax.set_xlabel('Comuna')
ax.set_ylabel('Matricula total')
ax.set_title('Matricula total por comuna - Cali 2026')
fmt_eje(ax, 'y')
plt.xticks(rotation=45, ha='right')
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h+200,
            fmt_num(h), ha='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Grafico: Proporcion oficial vs no oficial por comuna
fig, ax = plt.subplots(figsize=(12, 6))
pct_of = df_cu['Oficial'] / df_cu['Total'] * 100
pct_no = df_cu['No oficial'] / df_cu['Total'] * 100
ax.barh(df_cu['comuna'], pct_of, label='% Oficial', color='#66c2a5')
ax.barh(df_cu['comuna'], pct_no, left=pct_of, label='% No oficial', color='#fc8d62')
ax.set_xlabel('Porcentaje (%)')
ax.set_title('Composicion sectorial por comuna (Oficial vs No oficial)')
ax.axvline(50, color='gray', linestyle='--', alpha=0.5)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

---
## 2. Cruzar sedes con comuna (info geografica)

In [ ]:
# Cargar detalle por sede
ruta_doc = os.path.join(DATA_PATH, 'Indicadores de docentes y equipos de computo',
                        '03_Estudiantes_por_docente_y_equipo_2026.xlsx')
df_sede = pd.read_excel(ruta_doc, sheet_name='Detalle por sede')
print(f'Sedes con datos: {len(df_sede)}')

# Cargar info geografica
ruta_geo = os.path.join(DATA_PATH, 'Informaci\u00f3n geogr\u00e1fica sedes.xlsx')
if not os.path.exists(ruta_geo):
    ruta_geo = os.path.join(DATA_PATH, 'info_geografica',
                            'Informaci\u00f3n geogr\u00e1fica_sedes_Educativas.xlsx')
df_geo = pd.read_excel(ruta_geo)
print(f'Sedes en geo: {len(df_geo)}')

# Cruzar por codigo DANE
df_geo_min = df_geo[['EeCodDane', 'EEComCor']].copy()
df_geo_min['EeCodDane'] = df_geo_min['EeCodDane'].astype(str).str.strip()
df_sede['cod_dane_sede'] = df_sede['cod_dane_sede'].astype(str).str.strip()

df_sc = df_sede.merge(df_geo_min, left_on='cod_dane_sede',
                      right_on='EeCodDane', how='left')
match = df_sc['EEComCor'].notna().sum()
print(f'Sedes con comuna asignada: {match} de {len(df_sc)} ({match/len(df_sc)*100:.1f}%)')

# Filtrar comunas urbanas
df_sc = df_sc[df_sc['EEComCor'].between(1, 22)].copy()
df_sc.rename(columns={'EEComCor': 'num_comuna'}, inplace=True)
df_sc['num_comuna'] = df_sc['num_comuna'].astype(int)
print(f'Sedes en comunas urbanas: {len(df_sc)}')

In [ ]:
# Agregar indicadores por comuna
agg = df_sc.groupby('num_comuna').agg(
    sedes=('cod_dane_sede', 'count'),
    matricula_oficial=('matricula', 'sum'),
    equipos_total=('equipos', 'sum'),
    est_equipo_prom=('estudiantes_por_equipo', 'mean'),
    est_equipo_med=('estudiantes_por_equipo', 'median'),
).reset_index()

agg['est_equipo_calc'] = (agg['matricula_oficial'] / agg['equipos_total']).round(2)

# Merge con matricula total por comuna
df_final = df_cu[['num_comuna','Oficial','No oficial','Total','pct_oficial']].merge(
    agg, on='num_comuna', how='left')

print(f'Tabla consolidada: {df_final.shape}')
df_final

---
## 3. Graficos de indicadores por comuna

In [ ]:
# Grafico: Estudiantes por equipo de computo (promedio por comuna)
fig, ax = plt.subplots(figsize=(12, 5))
comunas_label = [f'C{i}' for i in df_final['num_comuna']]
colors = ['#e78ac3' if v > 10 else '#66c2a5' for v in df_final['est_equipo_prom']]
bars = ax.bar(comunas_label, df_final['est_equipo_prom'], color=colors)
ax.axhline(5, color='green', linestyle='--', alpha=0.7, label='Meta: 5 est/equipo')
ax.axhline(10, color='orange', linestyle='--', alpha=0.7, label='Alerta: 10 est/equipo')
ax.set_xlabel('Comuna')
ax.set_ylabel('Estudiantes por equipo')
ax.set_title('Promedio de estudiantes por equipo de computo - por comuna')
ax.legend()
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h+0.3, f'{h:.1f}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Grafico: Equipos totales por comuna
fig, ax = plt.subplots(figsize=(12, 5))
comunas_label = [f'C{i}' for i in df_final['num_comuna']]
bars = ax.bar(comunas_label, df_final['equipos_total'], color=sns.color_palette('Set2'))
ax.set_xlabel('Comuna')
ax.set_ylabel('Equipos de computo')
ax.set_title('Total equipos de computo en sedes oficiales - por comuna')
fmt_eje(ax, 'y')
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h+20, fmt_num(h), ha='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Grafico: Sedes oficiales por comuna
fig, ax = plt.subplots(figsize=(12, 5))
comunas_label = [f'C{i}' for i in df_final['num_comuna']]
bars = ax.bar(comunas_label, df_final['sedes'], color='#8da0cb')
ax.set_xlabel('Comuna')
ax.set_ylabel('Cantidad de sedes')
ax.set_title('Sedes educativas oficiales por comuna')
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h+0.3, f'{int(h)}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 4. Analisis de correlacion

In [ ]:
# Seleccionar variables numericas para correlacion
cols_corr = ['Total', 'Oficial', 'No oficial', 'pct_oficial',
             'sedes', 'matricula_oficial', 'equipos_total',
             'est_equipo_prom', 'est_equipo_calc']
df_corr = df_final[cols_corr].dropna()

# Matriz de correlacion
corr = df_corr.corr()

# Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, vmin=-1, vmax=1,
            cbar_kws={'label': 'Correlacion'})
ax.set_title('Matriz de correlacion - Indicadores educativos por comuna')
plt.tight_layout()
plt.show()

print('\nCorrelaciones mas fuertes (|r| > 0.6):')
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        r = corr.iloc[i, j]
        if abs(r) > 0.6:
            print(f'  {corr.columns[i]} vs {corr.columns[j]}: r = {r:.3f}')

In [ ]:
# Scatter: Matricula total vs Equipos
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Matricula vs Equipos
axes[0].scatter(df_final['Total'], df_final['equipos_total'], s=60, alpha=0.7)
axes[0].set_xlabel('Matricula total')
axes[0].set_ylabel('Equipos de computo')
axes[0].set_title('Matricula vs Equipos')
fmt_eje(axes[0], 'x')

# 2. Sedes vs Matricula oficial
axes[1].scatter(df_final['sedes'], df_final['matricula_oficial'], s=60, alpha=0.7, color='#fc8d62')
axes[1].set_xlabel('Sedes oficiales')
axes[1].set_ylabel('Matricula oficial')
axes[1].set_title('Sedes vs Matricula oficial')
fmt_eje(axes[1], 'y')

# 3. % Oficial vs Est/equipo
axes[2].scatter(df_final['pct_oficial'], df_final['est_equipo_prom'], s=60, alpha=0.7, color='#8da0cb')
axes[2].set_xlabel('% Matricula oficial')
axes[2].set_ylabel('Est. por equipo (prom)')
axes[2].set_title('% Oficial vs Ratio est/equipo')

plt.tight_layout()
plt.show()

---
## 5. Evaluacion de viabilidad del ICET

### Datos disponibles por comuna para calcular un indice:

| Dimension ICET | Indicador disponible | Fuente | Estado |
|---|---|---|---|
| Cobertura y Acceso (30%) | Matricula total, oficial, no oficial por comuna | 01_Matricula_2026 | Disponible |
| Cobertura y Acceso (30%) | Cobertura bruta y neta | 02_Indicadores_2026 | Solo nivel municipal |
| Eficiencia Interna (25%) | Repitencia por sector y nivel | 02_Indicadores_2026 | Solo nivel municipal |
| Recursos Pedagogicos (20%) | Estudiantes/equipo por sede -> agregable a comuna | 03_Estudiantes_2026 | Disponible |
| Infraestructura (15%) | Obras historicas, emprestito | Intervenciones | Disponible (por sede) |
| Dotacion Tecnologica (10%) | Equipos por sede -> agregable a comuna | 03_Estudiantes_2026 | Disponible |

### Conclusion:

**SI es viable calcular un indice por comuna** con los datos actuales, con las siguientes consideraciones:

1. **Cobertura (30%):** Se puede usar matricula total por comuna como proxy. Para cobertura bruta/neta
   se necesitaria poblacion por comuna en edad escolar (dato DANE disponible externamente).

2. **Eficiencia (25%):** Repitencia solo esta a nivel municipal. Se podria usar un valor constante
   para todas las comunas (referente municipal) o buscar el desglose por sede en SIMAT.

3. **Recursos (20%):** Estudiantes/equipo ya esta disponible por sede y se agrega a comuna sin problema.

4. **Infraestructura (15%):** Se puede contar intervenciones por sede y agregar a comuna.

5. **Dotacion (10%):** Equipos por sede disponibles, se agregan a comuna directamente.

### Recomendacion:

Calcular un **ICET parcial** con las 3 dimensiones que tienen datos completos por comuna:
- Cobertura (matricula) + Recursos (est/equipo) + Dotacion (equipos)
- Reponderando los pesos para que sumen 100%.
- Agregar Eficiencia e Infraestructura cuando se consiga el desglose por comuna/sede.

In [ ]:
# Resumen de datos disponibles por dimension
print('=' * 60)
print('RESUMEN DE VIABILIDAD DEL ICET POR COMUNA')
print('=' * 60)
print(f'Comunas urbanas con datos: {len(df_final)}')
print(f'Sedes oficiales mapeadas: {len(df_sc)}')
print(f'\nIndicadores disponibles por comuna:')
print(f'  - Matricula total:        22 comunas')
print(f'  - Matricula oficial:       22 comunas')
print(f'  - % sector oficial:        22 comunas')
print(f'  - Sedes oficiales:         22 comunas')
print(f'  - Equipos de computo:      22 comunas')
print(f'  - Est/equipo promedio:     22 comunas')
print(f'  - Cobertura bruta/neta:    Solo municipal')
print(f'  - Repitencia/desercion:    Solo municipal')
print(f'\nDimensiones calculables hoy:')
print(f'  Cobertura (proxy matricula):  SI')
print(f'  Recursos pedagogicos:         SI')
print(f'  Dotacion tecnologica:         SI')
print(f'  Eficiencia interna:           NO (solo municipal)')
print(f'  Infraestructura:              PARCIAL (requiere cruce)')
print(f'\nViabilidad: ICET parcial (3 de 5 dimensiones) = VIABLE')